# Bernoulli and binomial distributions
The Bernoulli PMF in one formula, the three-viewer example by counting and by formula, product purchases, and simulated binomial experiments.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from itertools import product
from math import comb
from scipy import stats

## 1. The Bernoulli PMF in one formula

In [ ]:
def bernoulli_pmf(x, p):
    # p^x (1 - p)^(1 - x): gives p for x = 1 and 1 - p for x = 0
    return p ** x * (1 - p) ** (1 - x)

print("coin, x = 1:", bernoulli_pmf(1, 0.5), " x = 0:", bernoulli_pmf(0, 0.5))
print("die shows 5, x = 1:", round(bernoulli_pmf(1, 1 / 6), 4), " x = 0:", round(bernoulli_pmf(0, 1 / 6), 4))
# scipy has the same distribution
print(stats.bernoulli.pmf([0, 1], 1 / 6))

## 2. Three viewers, each likes with probability 0.5: counting the sample space

In [ ]:
# every outcome is a string such as "LNL" (L = like, N = no like)
outcomes = ["".join(o) for o in product("LN", repeat=3)]
print(len(outcomes), outcomes)
for k in range(4):
    hits = [o for o in outcomes if o.count("L") == k]
    print(k, "likes:", hits, f"{len(hits)}/8")

## 3. The same answers from the binomial formula

In [ ]:
for k in range(4):
    by_hand = comb(3, k) * 0.5 ** k * 0.5 ** (3 - k)
    print(k, "likes:", by_hand, " scipy:", stats.binom.pmf(k, 3, 0.5))

## 4. Product purchases: shown to 1000 people, each buys with probability 0.1

In [ ]:
buyers = stats.binom(n=1000, p=0.1)
print("P(exactly 50) =", buyers.pmf(50))
# 'at least 70' = 1 - P(at most 69); sf(69) does this in one call
print("P(at least 70) =", round(buyers.sf(69), 4))
print("P(exactly 100) =", round(buyers.pmf(100), 4))
print("mean", buyers.mean(), "standard deviation", round(buyers.std(), 2))

## 5. Simulating 1000 binomial experiments of 10 coin tosses

In [ ]:
rng = np.random.default_rng(42)        # fixed seed
heads = rng.binomial(n=10, p=0.5, size=1000)
print(heads[:10])                       # heads in the first 10 experiments
print("counts of 0..10 heads:", np.bincount(heads, minlength=11))
print("average", heads.mean(), " variance", round(heads.var(), 4))

## 6. How p moves the shape

In [ ]:
fig = go.Figure()
for p in [0.1, 0.5, 0.8]:
    sim = np.random.default_rng(42).binomial(10, p, 1000)
    fig.add_histogram(x=sim, histnorm="probability", name=f"p = {p}", opacity=0.6, xbins=dict(start=-0.5, end=10.5, size=1))
    print(f"p = {p}: average {sim.mean():.3f}, n*p = {10 * p}, skewness {float(stats.binom(10, p).stats(moments='s')):.3f}")
fig.update_layout(barmode="overlay", template="simple_white", xaxis_title="heads in 10 tosses")
fig.show()

## 7. More runs, closer bars
Largest gap between the simulated share and the exact PMF, for 100, 1000 and 100,000 runs.

In [ ]:
for p in [0.1, 0.5, 0.8]:
    exact = stats.binom(10, p).pmf(np.arange(11))
    for runs in [100, 1000, 100_000]:
        sim = np.bincount(np.random.default_rng(42).binomial(10, p, runs), minlength=11) / runs
        print(f"p = {p}, {runs} runs: largest gap {abs(sim - exact).max():.4f}")
